In [7]:
from dotenv import load_dotenv
import os
import boto3
import pandas as pd

load_dotenv()

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{os.environ['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
    aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
    region_name="auto",
)

bucket = "colombia-dengue-forecasting"

In [8]:
years = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]

cases_data_frames = []

for year in years:
    key = f"raw/cases/Datos_{year}_210.xlsx"
    local_path = f"data/raw/cases/{year}.xlsx"

    s3.download_file(bucket, key, local_path)
    df = pd.read_excel(local_path)
    cases_data_frames.append(df)

In [9]:
cases_data_frames[0].info()    

<class 'pandas.DataFrame'>
RangeIndex: 100117 entries, 0 to 100116
Data columns (total 69 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   CONSECUTIVE                100117 non-null  int64  
 1   COD_EVE                    100117 non-null  int64  
 2   FEC_NOT                    100117 non-null  str    
 3   SEMANA                     100117 non-null  int64  
 4   ANO                        100117 non-null  int64  
 5   COD_PRE                    100117 non-null  int64  
 6   COD_SUB                    100117 non-null  int64  
 7   EDAD                       100117 non-null  int64  
 8   UNI_MED                    100117 non-null  int64  
 9   SEXO                       100117 non-null  str    
 10  COD_PAIS_O                 100117 non-null  int64  
 11  COD_DPTO_O                 100117 non-null  int64  
 12  COD_MUN_O                  100117 non-null  int64  
 13  AREA                       100117 non-nu

| Column | Meaning | Values / format |
|---|---|---|
| `COD_EVE` | SIVIGILA event code | `210` = dengue; `220` = severe dengue; `580` = dengue mortality, handled as a separate event |
| `ANO` | Epidemiological year | The years present in the dataset |
| `SEMANA` | Epidemiological week | Week number, generally `1`–`53`; use the week paired with `ANO` |
| `COD_DPTO_O` | Department where the case occurred | DANE DIVIPOLA department code; keep as text to preserve leading zeros |
| `COD_MUN_O` | Municipality where the case occurred | DANE DIVIPOLA municipality code; check whether this export stores the municipality component or the full code |
| `COD_PAIS_O` | Country where the case occurred | Country code; Colombia is commonly represented as `170` |
| `INI_SIN` | Date symptoms began | Date; parse using the format in your source file |
| `FEC_NOT` | Date the case was notified | Date; parse using the format in your source file |
| `TIP_CAS` | Initial case classification/type | Event- and year-specific code; check the dengue data dictionary and the values in your files |
| `AJUSTE` | Case adjustment or updated classification | Dengue protocol mentions `3` = confirmed by laboratory, `5` = confirmed by epidemiological link, `6` = discarded, and `D` = digitization error; verify other values by year |
| `CONSECUTIVE` | Sequential record identifier | Identifier, not a category; uniqueness is not guaranteed across separate yearly files |

In [ ]:
cols_to_keep = ["COD_EVE", "ANO", "SEMANA", "COD_DPTO_O", "COD_MUN_O", "COD_PAIS_O", "INI_SIN", "FEC_NOT", "TIP_CAS", "AJUSTE", "CONSECUTIVE"]
for i in range(0,len(cases_data_frames)):
    cases_data_frames[i] = cases_data_frames[i][cols_to_keep]

cases_data_frames[0].head()

,COD_EVE,ANO,SEMANA,COD_DPTO_O,COD_MUN_O,COD_PAIS_O,INI_SIN,FEC_NOT,TIP_CAS,AJUSTE,CONSECUTIVE
0,210,2016,21,5,1,170,25/05/2016 12:00:00 a. m.,1/06/2016 12:00:00 a. m.,3,0,713315
1,210,2016,3,5,1,170,18/01/2016 12:00:00 a. m.,21/01/2016 12:00:00 a. m.,3,0,713316
2,210,2016,2,5,1,170,15/01/2016 12:00:00 a. m.,23/01/2016 12:00:00 a. m.,2,0,713323
3,210,2016,22,5,1,170,1/06/2016 12:00:00 a. m.,11/06/2016 12:00:00 a. m.,3,0,713353
4,210,2016,25,5,1,170,20/06/2016 12:00:00 a. m.,25/06/2016 12:00:00 a. m.,2,0,713310


| Column | Value | Meaning |
|---|---:|---|
| `TIP_CAS` | `2` | Probable case |
| `TIP_CAS` | `3` | Confirmed by laboratory |
| `TIP_CAS` | `5` | Confirmed by epidemiological link |
| `AJUSTE` | `0` | No adjustment / first report |
| `AJUSTE` | `3` | Adjustment confirming by laboratory |
| `AJUSTE` | `5` | Adjustment confirming by epidemiological link |
| `AJUSTE` | `7` | Other update to the record; not a case classification |